# Baseline Model

This notebook trains a baseline model using EfficientNet with its backbone frozen — only the final classification layer is replaced and trained from scratch for 525 bird species.

The purpose of this baseline is to establish a reference accuracy before moving to full fine-tuning. By training only the head, we can quickly see what performance the pretrained ImageNet features alone can achieve on our dataset.

**Optimizer:** SGD, lr=0.01, momentum=0.9  
**Scheduler:** StepLR — reduce LR by 0.1 every 5 epochs (drops at epoch 5 and 10)  
**Epochs:** 15  
**What is trained:** Final classification layer only (backbone weights are frozen)  
**Experiment tracking:** MLflow  
**Metrics:** Top-1 accuracy, Top-5 accuracy  
**Goal:** Establish a baseline to beat with full fine-tuning

## 0. Sync Data from S3

Downloads the dataset from S3 to the SageMaker instance's local storage. This only runs once per session — `aws s3 sync` skips files that already exist.

In [ ]:
import subprocess
from pathlib import Path

S3_BUCKET    = "bird-ml-halajeel"
S3_DATA_PATH = "data/raw/birds-525"
LOCAL_DATA_DIR = Path("./data/raw/birds-525")

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run([
    "aws", "s3", "sync",
    f"s3://{S3_BUCKET}/{S3_DATA_PATH}/",
    str(LOCAL_DATA_DIR),
], check=True)

print(f"Data ready at: {LOCAL_DATA_DIR.resolve()}")

## 1. Load Dataset & DataLoaders

In [ ]:
from datasets import load_dataset
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import pyarrow.parquet as pq

RAW_DATA_DIR = Path("./data/raw/birds-525")

dataset = load_dataset("parquet", data_dir=str(RAW_DATA_DIR / "data"))


In [ ]:

# Labels in the dataset are sparse — build a mapping from original label to dense index
# by scanning all parquet files directly with pyarrow (much faster than going through HuggingFace).
all_labels = set()
for parquet_file in (RAW_DATA_DIR / "data").glob("*.parquet"):
    table = pq.read_table(parquet_file, columns=["label"])
    all_labels.update(table["label"].to_pylist())

label_to_idx = {original: dense for dense, original in enumerate(sorted(all_labels))}
print(f"Built label remapping for {len(label_to_idx)} classes (dense range 0–{len(label_to_idx) - 1})")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class BirdDataset(Dataset):
    def __init__(self, hf_split, transform, label_to_idx):
        self.data = hf_split
        self.transform = transform
        self.label_to_idx = label_to_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        return self.transform(sample["image"]), self.label_to_idx[sample["label"]]

BATCH_SIZE = 512

train_loader = DataLoader(BirdDataset(dataset["train"],      train_transforms, label_to_idx), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(BirdDataset(dataset["validation"], eval_transforms,  label_to_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(BirdDataset(dataset["test"],       eval_transforms,  label_to_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoaders ready.")

## 2. Load EfficientNet-B3 & Modify Head

In [ ]:
import torch
from torchvision import models
from pathlib import Path

NUM_CLASSES  = len(label_to_idx)  # actual number of classes from the remapping
WEIGHTS_PATH = Path("./efficientnet_b3_rwightman-b3899882.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Number of classes: {NUM_CLASSES}")

# Create the model architecture without downloading weights
model = models.efficientnet_b3(weights=None)

# Load the pretrained weights from the local .pth file and apply them
state_dict = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=True)
model.load_state_dict(state_dict)

# Freeze all backbone parameters — no gradients will be computed for them
for param in model.parameters():
    param.requires_grad = False

# Replace the final classifier with a new linear layer for NUM_CLASSES classes.
# model.classifier is a Sequential(Dropout, Linear) — we only replace the Linear part.
in_features = model.classifier[1].in_features
model.classifier[1] = torch.nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 3. Loss, Optimizer & Scheduler

In [5]:
import torch.nn as nn
import torch.optim as optim

EPOCHS    = 15
LR        = 0.01
MOMENTUM  = 0.9
STEP_SIZE = 5
GAMMA     = 0.1

# CrossEntropyLoss is the standard loss for multi-class classification
criterion = nn.CrossEntropyLoss()

# SGD only optimizes the trainable parameters (the new classifier head)
optimizer = optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    momentum=MOMENTUM,
)

# StepLR multiplies the LR by gamma every step_size epochs
# LR drops: epoch 5 → 0.001, epoch 10 → 0.0001
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)

print(f"Criterion: {criterion}")
print(f"Optimizer: {optimizer}")
print(f"Scheduler: {scheduler}")

Criterion: CrossEntropyLoss()
Optimizer: SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    initial_lr: 0.01
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0
)
Scheduler: <torch.optim.lr_scheduler.StepLR object at 0x0000018387B5E510>


## 4. Training Loop

In [6]:
images, labels = next(iter(train_loader))
print(f"Image shape: {images.shape}, dtype: {images.dtype}")
print(f"Labels min: {labels.min().item()}, max: {labels.max().item()}, dtype: {labels.dtype}")


Image shape: torch.Size([512, 3, 224, 224]), dtype: torch.float32
Labels min: 0, max: 525, dtype: torch.int64


In [ ]:
import mlflow
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

def top_k_accuracy(outputs, labels, k):
    _, top_k_preds = outputs.topk(k, dim=1)
    correct = top_k_preds.eq(labels.view(-1, 1).expand_as(top_k_preds))
    return correct.any(dim=1).float().mean().item()

def run_epoch(loader, training, epoch_label):
    model.train() if training else model.eval()
    total_loss, top1, top5 = 0.0, 0.0, 0.0

    pbar = tqdm(loader, desc=epoch_label, leave=False)
    with torch.set_grad_enabled(training):
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            top1 += top_k_accuracy(outputs, labels, k=1)
            top5 += top_k_accuracy(outputs, labels, k=5)

            pbar.set_postfix(loss=f"{loss.item():.4f}")

    n = len(loader)
    return total_loss / n, top1 / n, top5 / n


# Lists to accumulate per-epoch metrics for plotting after training completes
train_losses, val_losses = [], []

# mlflow.start_run() opens a new experiment run and returns a context manager.
# All params and metrics logged inside the block are attached to this run.
mlflow.set_experiment("baseline")

with mlflow.start_run(run_name="efficientnet_b3_head_only"):

    # Log all hyperparameters for this run
    mlflow.log_params({
        "model": "efficientnet_b3",
        "epochs": EPOCHS,
        "lr": LR,
        "momentum": MOMENTUM,
        "step_size": STEP_SIZE,
        "gamma": GAMMA,
        "batch_size": BATCH_SIZE,
        "trained_layers": "head_only",
    })

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_top1, train_top5 = run_epoch(train_loader, training=True,  epoch_label=f"Epoch {epoch}/{EPOCHS} [train]")
        val_loss,   val_top1,   val_top5   = run_epoch(val_loader,   training=False, epoch_label=f"Epoch {epoch}/{EPOCHS} [val]  ")
        scheduler.step()

        # Track losses for the plot
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # Log metrics for this epoch — MLflow stores one value per step per metric
        mlflow.log_metrics({
            "train_loss":  train_loss,
            "train_top1":  train_top1,
            "train_top5":  train_top5,
            "val_loss":    val_loss,
            "val_top1":    val_top1,
            "val_top5":    val_top5,
            "lr":          scheduler.get_last_lr()[0],
        }, step=epoch)

        print(f"Epoch {epoch:02d}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | Train Top-1: {train_top1:.4f} | Train Top-5: {train_top5:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Top-1: {val_top1:.4f} | Val Top-5: {val_top5:.4f}")

    # Log the final model checkpoint as an MLflow artifact
    mlflow.pytorch.log_model(model, artifact_path="model")
    print("\nRun complete. Model logged to MLflow.")

# Plot the loss curves
epochs_range = range(1, EPOCHS + 1)
plt.figure(figsize=(10, 5))
plt.plot(epochs_range, train_losses, marker="o", label="Train loss")
plt.plot(epochs_range, val_losses,   marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Average loss")
plt.title("Training vs Validation loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Evaluate on Test Set

Run the trained model on the held-out test set to get a final estimate of generalization performance.

In [ ]:
test_loss, test_top1, test_top5 = run_epoch(test_loader, training=False, epoch_label="Test")

print(f"Test Loss:    {test_loss:.4f}")
print(f"Test Top-1:   {test_top1:.4f}")
print(f"Test Top-5:   {test_top5:.4f}")

# Log test metrics to the same MLflow run we used during training
with mlflow.start_run(run_id=mlflow.last_active_run().info.run_id):
    mlflow.log_metrics({
        "test_loss": test_loss,
        "test_top1": test_top1,
        "test_top5": test_top5,
    })